# Model 2 — Söz Yazarı (Lyrics Generator) — V2
`{emotion, bpm, key, instruments, vocal_style}` → Türkçe şarkı sözü

V2 dataset: orijinal 342 + Ollama augmented 3720 = ~4062 örnek

Başlamadan önce:
1. Sağ panel → **Add Data** → **Upload** → `dataset_lyrics_v2.jsonl`
2. Accelerator: **GPU T4 x1** (T4 x2 seçme — DataParallel gather OOM yapar)
3. Hücreleri sırayla çalıştır

In [ ]:
# ── 1. Kurulum ────────────────────────────────────────────────────────────────
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
# HF cache'i /tmp'e yönlendir — /kaggle/working'e gitmez, ~2GB tasarruf
os.environ["HF_HOME"] = "/tmp/hf_cache"

!pip install -q transformers datasets accelerate sentencepiece protobuf
print('Kurulum tamamlandi.')

In [ ]:
# ── 2. Dataset: V2 (orijinal + augmented) ───────────────────────────────────
# dataset_lyrics_v2.jsonl zaten önceden formatlanmış: input_text + target_text
import glob, json
from datasets import Dataset

candidates = glob.glob('/kaggle/input/**/dataset_lyrics_v2.jsonl', recursive=True)
RAW_PATH = candidates[0] if candidates else '/kaggle/working/dataset_lyrics_v2.jsonl'
print(f'Dataset: {RAW_PATH}')

records = []
with open(RAW_PATH, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        d = json.loads(line)
        target = d.get('target_text', '').strip()
        # 20 kelimeden kısa olanları atla (güvenlik filtresi)
        if not target or len(target.split()) < 20:
            continue
        records.append({
            'input_text':  d['input_text'],
            'target_text': target,
        })

ds = Dataset.from_list(records)
split = ds.train_test_split(test_size=0.10, seed=42)
train_ds, eval_ds = split['train'], split['test']
print(f'Kayit: {len(records)} | Egitim: {len(train_ds)} | Dogrulama: {len(eval_ds)}')
print('Ornek girdi:', records[0]['input_text'])
print('Ornek cikti (ilk 150):', records[0]['target_text'][:150])

In [ ]:
# ── 3. Konfigürasyon ──────────────────────────────────────────────────────────
BASE_MODEL     = 'google/mt5-small'
OUTPUT_DIR     = '/kaggle/working/story-to-music-lyricist'
CHECKPOINT_DIR = '/kaggle/working/checkpoints-lyricist'

MAX_INPUT_LEN  = 64
MAX_TARGET_LEN = 384   # tam verse+chorus görsün

# V2 dataset: ~4062 örnek (13× büyük), daha az epoch yeterli
EPOCHS      = 12       # 40→12: 13× veri var, ~aynı toplam gradient update
BATCH_SIZE  = 1        # fp32 + 384 token belleğe sığsın
GRAD_ACCUM  = 16       # efektif batch = 16
LR          = 1e-4     # mt5 fine-tuning için stabil
WARMUP_RATIO = 0.10    # daha çok step → daha kısa warmup ratio yeter
WEIGHT_DECAY = 0.01

print('Konfigurasyon tamam.')

In [ ]:
# ── 4. Model ve Tokenizer ─────────────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cihaz: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)
# gradient_checkpointing_enable() buradan KALDIRILDI:
# Trainer, gradient_checkpointing=True argümanıyla bunu içten açar ama
# training bittikten sonra KAPATMAZ — generate() sırasında torch.no_grad()
# ile çakışarak garbage token (<0x03>) üretir.
print(f'Parametre sayisi: {model.num_parameters():,}')

In [ ]:
# ── 5. Tokenizasyon ───────────────────────────────────────────────────────────
from transformers import DataCollatorForSeq2Seq

def tokenize(batch):
    model_inputs = tokenizer(
        batch['input_text'],
        max_length=MAX_INPUT_LEN,
        truncation=True,
        padding=False,
    )
    labels = tokenizer(
        text_target=batch['target_text'],
        max_length=MAX_TARGET_LEN,
        truncation=True,
        padding=False,
    )
    model_inputs['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in ids]
        for ids in labels['input_ids']
    ]
    return model_inputs

train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
eval_tok  = eval_ds.map(tokenize,  batched=True, remove_columns=eval_ds.column_names)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8
)
print('Tokenizasyon tamam.')

In [ ]:
# ── 6. Eğitim ─────────────────────────────────────────────────────────────────
from pathlib import Path
import transformers
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, TrainerCallback

class PrintLossCallback(TrainerCallback):
    """Loss değerlerini Kaggle loglarına bastır — fp16 NaN tespiti için."""
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        if 'loss' in logs:
            print(f'Epoch {state.epoch:.1f} | Step {state.global_step} | Train Loss: {logs["loss"]:.4f}')
        if 'eval_loss' in logs:
            print(f'Epoch {state.epoch:.1f} | Eval  Loss: {logs["eval_loss"]:.4f}')

total_steps  = (len(train_tok) // (BATCH_SIZE * GRAD_ACCUM)) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    warmup_steps=warmup_steps,
    lr_scheduler_type='cosine',
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=False,
    save_total_limit=1,
    fp16=False,                    # fp16 mt5-small ağırlıklarını bozuyor (NaN)
    gradient_checkpointing=True,
    predict_with_generate=False,
    logging_steps=10,
    logging_first_step=True,
    report_to='none',
    seed=42,
)

trainer_kwargs = dict(
    model=model, args=args,
    train_dataset=train_tok, eval_dataset=eval_tok,
    data_collator=data_collator,
    callbacks=[PrintLossCallback()],
)
ver = tuple(int(x) for x in transformers.__version__.split('.')[:2])
trainer_kwargs['processing_class' if ver >= (4, 46) else 'tokenizer'] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

checkpoints = sorted(Path(CHECKPOINT_DIR).glob('checkpoint-*'), key=lambda x: int(x.name.split('-')[-1])) if Path(CHECKPOINT_DIR).exists() else []
last_ckpt = str(checkpoints[-1]) if checkpoints else None
if last_ckpt:
    print(f'Checkpoint bulundu: {last_ckpt}')

trainer.train(resume_from_checkpoint=last_ckpt)

In [ ]:
# ── 7. Modeli Kaydet ──────────────────────────────────────────────────────────
import shutil

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Checkpoints artık gereksiz, diski boşalt
if Path(CHECKPOINT_DIR).exists():
    shutil.rmtree(CHECKPOINT_DIR)
    print('Checkpoints temizlendi.')

total_mb = sum(f.stat().st_size for f in Path(OUTPUT_DIR).rglob('*') if f.is_file()) / (1024**2)
print(f'Model kaydedildi: {OUTPUT_DIR}  ({total_mb:.1f} MB)')

In [ ]:
# ── 8. Smoke Test ─────────────────────────────────────────────────────────────
model.gradient_checkpointing_disable()
model.eval()

# NaN kontrolü — fp16 bozulmasını doğrula
nan_count = sum(p.isnan().any().item() for p in model.parameters())
print(f'NaN içeren parametre tensörü: {nan_count}  (0 olmalı)')

ornek_input = 'Türkçe şarkı sözü yaz: duygu=hüzün, enerji=4, tempo=72 BPM, ton=A minor, enstrümanlar=ney, keman, piyano, vokal=erkek, kısık, dramatik'
print('GIRDI:', ornek_input)

inputs = tokenizer(ornek_input, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=350,
        num_beams=4,
        no_repeat_ngram_size=3,          # dejenere loop önle
        repetition_penalty=1.4,          # tekrar cezalandır
        early_stopping=True,
    )

print('Token IDs:', out[0].tolist()[:20], '...')
decoded = tokenizer.decode(out[0], skip_special_tokens=True)
print('\nCIKTI:')
print(decoded)

In [ ]:
# ── 9. Değerlendirme ──────────────────────────────────────────────────────────
import random
from collections import Counter

TURKISH_CHARS = set('çğıöşüÇĞİÖŞÜ')

def turkce_skor(text):
    if not text:
        return 0.0
    tr = sum(1 for c in text if c in TURKISH_CHARS)
    alpha = sum(1 for c in text if c.isalpha())
    return tr / alpha if alpha > 0 else 0.0

def ortalama_kelime_uzunlugu(text):
    words = [w for w in text.split() if w.isalpha()]
    return sum(len(w) for w in words) / len(words) if words else 0

def tekrar_orani(text):
    """En çok tekrar eden kelimenin oranı — 0.3 üstü dejenere loop."""
    words = [w.lower() for w in text.split() if w.isalpha()]
    if len(words) < 10:
        return 0.0
    counts = Counter(words)
    return counts.most_common(1)[0][1] / len(words)

model.gradient_checkpointing_disable()
model.eval()

GEN_KWARGS = dict(
    max_new_tokens=350,
    num_beams=4,
    no_repeat_ngram_size=3,
    repetition_penalty=1.4,
    early_stopping=True,
)

random.seed(0)
samples = random.sample(range(len(eval_ds)), min(30, len(eval_ds)))
has_tags, long_enough, tr_ok, avg_wl_ok, no_loop = 0, 0, 0, 0, 0

for idx in samples:
    inp = eval_ds[idx]['input_text']
    inputs = tokenizer(inp, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, **GEN_KWARGS)
    gen = tokenizer.decode(out[0], skip_special_tokens=True)

    if '[Verse' in gen or '[Chorus' in gen:
        has_tags += 1
    if len(gen.split()) >= 30:
        long_enough += 1
    if turkce_skor(gen) >= 0.03:
        tr_ok += 1
    if ortalama_kelime_uzunlugu(gen) >= 4.0:
        avg_wl_ok += 1
    if tekrar_orani(gen) < 0.3:
        no_loop += 1

n = len(samples)
print(f'Yapısal tag      : {has_tags}/{n} ({has_tags/n:.0%})')
print(f'30+ kelime       : {long_enough}/{n} ({long_enough/n:.0%})')
print(f'Türkçe karakter  : {tr_ok}/{n} ({tr_ok/n:.0%})')
print(f'Kelime uzunluğu  : {avg_wl_ok}/{n} ({avg_wl_ok/n:.0%})')
print(f'Loop yok         : {no_loop}/{n} ({no_loop/n:.0%})  ← dejenerasyon')

# 3 örnek göster
print('\n── Örnek çıktılar ──')
for idx in samples[:3]:
    inp = eval_ds[idx]['input_text']
    inputs = tokenizer(inp, return_tensors='pt', max_length=MAX_INPUT_LEN, truncation=True).to(device)
    with torch.no_grad():
        out = model.generate(**inputs, **GEN_KWARGS)
    print('\nGIRDI :', inp[:80])
    print('CIKTI :', tokenizer.decode(out[0], skip_special_tokens=True)[:400])
    print('─' * 60)

if has_tags/n >= 0.70 and tr_ok/n >= 0.50 and no_loop/n >= 0.70:
    print('\n✅ Model 2 hazır.')
else:
    print('\n⚠️ Hâlâ sorun var.')

In [ ]:
# ── 10. İndir ─────────────────────────────────────────────────────────────────
import shutil
shutil.make_archive('/kaggle/working/story-to-music-lyricist', 'zip', OUTPUT_DIR)
print('Output panelinden story-to-music-lyricist.zip indirebilirsin.')